In [27]:
from langchain_openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
ai = OpenAI(model="gpt-5-mini", temperature=0.9)

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")


### Tools

In [10]:
from langchain.tools import tool


In [11]:
@tool

def tool_duckduckgo_search(query:str):
    """Use this tool when you want to answer questions about current events or general knowledge. The input to this tool should be a search query."""

    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    return search.invoke(query)
    

In [12]:
@tool

def tool_wikipedia_search(query:str):
    """Use this tool when you want to answer questions about general knowledge. The input to this tool should be a search query."""

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wikipedia.run(query)

In [13]:
tool_wikipedia_search.invoke("Sam altman")

'Page: Sam Altman\nSummary: Samuel Harris Altman (born April 22, 1985) is an American businessman and entrepreneur who has been the chief executive officer (CEO) of the artificial intelligence research organization OpenAI since 2019.\nAltman attended Stanford University for two years before he dropped out and co-founded Loopt, a smartphone geosocial networking service, which raised more than $30 million in venture capital before being acquired by Green Dot Corporation for $43.4 million in cash. In 2011, Altman joined Y Combinator, a technology startup accelerator and venture capital firm, and was the company\'s president from 2014 to 2019. \nAfter co-founding OpenAI in 2015, Altman later became the organization\'s CEO in 2019. In 2023, he was ousted by the organization\'s board of directors, who cited a lack of "confidence in his ability to continue leading OpenAI" in an official post. The move was met with significant backlash from employees and investors, resulting in Altman\'s reins

In [20]:
@tool
def tool_arxiv_search(query:str):
    """Use this tool when you want to answer questions about recent research papers. The input to this tool should be a search query."""
    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    arxiv_wrapper = ArxivAPIWrapper(
        top_k_results=3,
        doc_content_chars_max=2000
    )

    arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

    query="What are the recent papers published by Sam Altman"
    results = arxiv_tool.run(query)
    return results

In [19]:
def personal_info(query:str):
    """Use this tool when you want to answer questions about Sam Altman. The input to this tool should be a search query."""
    infos=[{
        "name":"Sam Altman",
        "age":42,
        "profession":"Entrepreneur",
        "current_position":"CEO of OpenAI",
        "education":"Stanford University"
    },
    { 
        "name":"Rahul Yadav",
        "age":25,
        "profession":"Software Engineer",
        "current_position":"Developer at Tech Corp",
        "education":"MIT"
    }]

    for info in infos:
        if query.lower() in info["name"].lower():
            return info
    return "No information found for the given query."

## Build Tools

In [25]:
toolkit=[
    tool_wikipedia_search,
    tool_duckduckgo_search,
    tool_arxiv_search,
    personal_info
]

In [28]:
llm_bind = llm.bind_tools(toolkit)


In [29]:
llm_bind.invoke("What are the recent research papers published by Sam Altman?")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 200, 'total_tokens': 218, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d2497e2d52', 'id': 'chatcmpl-DV957zt48MgOjjrfeAslBptdUvqZd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d94a6-b645-77c1-9759-cf6e0ed4bc79-0', tool_calls=[{'name': 'tool_arxiv_search', 'args': {'query': 'Sam Altman'}, 'id': 'call_eIwxW1KtZ4VI8oueKEtFo1G1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 200, 'output_tokens': 18, 'total_tokens': 218, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})